[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/28.%20The%20Integrated%20Crane%20Assignment%20%26%20Scheduling%20Problem%20%28QCAP-QCSP%29/P28-Tier-1_executed.ipynb)

---



# 28. The Integrated Crane Assignment and Scheduling Problem
## Tier 1: The Pen-and-Paper Method (Integer Programming Formulation)

### Goal and scope

This notebook formulates a small quay crane assignment and scheduling problem as a mixed-integer linear program (MILP). A discrete time period is one unit of processing time. The model assigns cranes to bays in each period, respects crane capacity, prevents cranes from crossing, enforces a minimum clearance between simultaneously active cranes, and minimizes the makespan.

The example contains two vessels, three cranes, four bays, and an eight-period planning horizon. The model permits a bay's work to be interrupted between periods, but at most one crane can process that bay in any one period. Crane travel times, setup times, precedence between bays, and changing crane positions while idle are outside the scope of this time-indexed model. Arrival times and due dates are stored in the input data but are not used as constraints or objective terms; all example vessels are available at time zero, and the objective is makespan rather than lateness.

### Assumptions

- Cranes have a fixed left-to-right order given by their order in the crane list. Their active assignments must respect that order, so cranes cannot cross.
- When two cranes are active in the same period, their assigned bay positions must be separated by at least the specified clearance distance.
- A bay requires its stated number of processing periods. It is processed by at most one crane in each period, though its work may be split across periods or cranes.
- Each crane processes at most one bay in each period.
- Assignments occupy whole periods. Period \(t\) is the interval \([t,t+1)\), so work in period \(t\) completes at time \(t+1\).

### Solution sequence

1. Define the vessels, bays, cranes, and planning horizon.
2. Create binary assignment variables and integer completion-related variables.
3. Minimize makespan subject to processing, capacity, spatial-order, clearance, and completion-time constraints.
4. Solve the MILP with PuLP's CBC solver, extract the schedule, and verify and interpret the results.
5. Visualize the schedule and summarize workload and utilization.

An exact solver can certify optimality when it reports an optimal status. A feasible incumbent found under a time limit is not, by itself, a proof of optimality.

## Mathematical model

The model makes a decision for every crane, bay, and time period: whether that crane processes that bay during that period. This time-indexed representation is useful here because processing requirements and capacity are expressed directly as counts of discrete periods.

Let \(V\) be the set of vessels, \(B_v\) the bays belonging to vessel \(v\), \(C\) the ordered set of cranes, and \(T=\{0,1,\ldots,H-1\}\) the set of time periods. Bay \((v,b)\) has position \(q_{vb}\) and requires \(p_{vb}\) periods of processing. Let \(d\) be the minimum clearance distance. The binary decision variable \(x_{cvbt}\) equals 1 if crane \(c\) processes bay \((v,b)\) in period \(t\), and 0 otherwise.

Each bay must receive its required amount of work, and it cannot be processed by two cranes at once:

$$
\sum_{c\in C}\sum_{t\in T} x_{cvbt}=p_{vb}
\qquad \forall v\in V,\ b\in B_v,
$$

$$
\sum_{c\in C}x_{cvbt}\leq 1
\qquad \forall v\in V,\ b\in B_v,\ t\in T.
$$

The first constraint counts all assigned processing periods for a bay. The second prevents simultaneous processing of that bay by multiple cranes. Each crane also has capacity for at most one bay per period:

$$
\sum_{v\in V}\sum_{b\in B_v}x_{cvbt}\leq 1
\qquad \forall c\in C,\ t\in T.
$$

To express the spatial rule, consider cranes \(c_1<c_2\) in their fixed left-to-right order and bays \(i=(v_1,b_1)\), \(j=(v_2,b_2)\). If assigning \(c_1\) to \(i\) and \(c_2\) to \(j\) would violate order or clearance, those two assignments cannot occur together:

$$
x_{c_1 i t}+x_{c_2 j t}\leq 1
\quad\text{whenever}\quad q_i+d>q_j,
\qquad \forall t\in T.
$$

Here, \(q_i\) and \(q_j\) are the physical positions of the bays. The condition excludes a reversed order and any separation smaller than \(d\). It applies when both cranes are active in the period; the model does not track the positions of idle cranes.

Let \(z_{vb}\) be a lower bound on the completion time of bay \((v,b)\), \(w_v\) a lower bound on vessel \(v\)'s completion time, and \(M\) the makespan. Since period \(t\) ends at time \(t+1\), the completion-related constraints are:

$$
z_{vb}\geq (t+1)x_{cvbt}
\qquad \forall v\in V,\ b\in B_v,\ c\in C,\ t\in T,
$$

$$
w_v\geq z_{vb}
\qquad \forall v\in V,\ b\in B_v,
\qquad
M\geq w_v
\qquad \forall v\in V.
$$

The objective is:

$$
\min M.
$$

The variable domains complete the formulation:

$$
x_{cvbt}\in\{0,1\},\qquad
z_{vb},w_v,M\in\mathbb{Z}_{\geq 0}.
$$

The completion variables are lower bounds: the constraints ensure that the makespan cannot be earlier than any assigned work. The schedule's actual bay and vessel completion times are therefore calculated from the extracted assignments, rather than interpreted as exact values of every lower-bound variable.

In the code, `assignment_vars` stores \(x\), `completion_vars` stores \(z\), `vessel_completion_vars` stores \(w\), and `makespan_var` stores \(M\). The constraint-building loops implement the equations above. The number of binary assignment variables grows as \(|C|\sum_v|B_v|H\); pairwise spatial restrictions can also grow with the number of crane pairs, bay pairs, and periods. This makes the formulation appropriate for small teaching instances, while larger instances may require substantial solver time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
import pulp
import seaborn as sns
from dataclasses import dataclass
from typing import List

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
@dataclass
class Bay:
    """A vessel bay with a physical position and processing requirement."""

    vessel_id: int
    bay_id: int
    position: int
    processing_time: int


@dataclass
class Vessel:
    """A vessel and its collection of bays."""

    vessel_id: int
    arrival_time: int
    due_date: int
    bays: List[Bay]


@dataclass
class Crane:
    """A quay crane with an initial position on the rail."""

    crane_id: int
    initial_position: int


class IntegratedQCASP:
    """Build, solve, and analyze the time-indexed crane assignment MILP."""

    def __init__(
        self,
        vessels: List[Vessel],
        cranes: List[Crane],
        time_horizon: int,
        clearance_distance: int,
    ):
        self.vessels = vessels
        self.cranes = cranes
        self.time_horizon = time_horizon
        self.clearance_distance = clearance_distance

        self.all_bays = [bay for vessel in vessels for bay in vessel.bays]
        self.n_vessels = len(vessels)
        self.n_cranes = len(cranes)
        self.n_bays = len(self.all_bays)

        self.model = None
        self.assignment_vars = {}
        self.completion_vars = {}
        self.vessel_completion_vars = {}
        self.makespan_var = None

    def create_mathematical_model(self):
        """Create the decision variables used by the integer program."""
        print("Creating the mathematical model...")

        self.model = pulp.LpProblem(
            "Integrated_QCASP", pulp.LpMinimize
        )

        # A binary variable records one crane-bay assignment in one period.
        self.assignment_vars = {}
        for c in range(self.n_cranes):
            for v_idx, vessel in enumerate(self.vessels):
                for bay in vessel.bays:
                    for t in range(self.time_horizon):
                        key = (c, v_idx, bay.bay_id, t)
                        var_name = f"x_{c}_{v_idx}_{bay.bay_id}_{t}"
                        self.assignment_vars[key] = pulp.LpVariable(
                            var_name, None, None, "Binary"
                        )

        # These lower-bound variables connect assigned work to makespan.
        self.completion_vars = {}
        for v_idx, vessel in enumerate(self.vessels):
            for bay in vessel.bays:
                key = (v_idx, bay.bay_id)
                var_name = f"z_{v_idx}_{bay.bay_id}"
                self.completion_vars[key] = pulp.LpVariable(
                    var_name, 0, None, "Integer"
                )

        self.vessel_completion_vars = {}
        for v_idx in range(self.n_vessels):
            var_name = f"w_{v_idx}"
            self.vessel_completion_vars[v_idx] = pulp.LpVariable(
                var_name, 0, None, "Integer"
            )

        self.makespan_var = pulp.LpVariable(
            "makespan", 0, None, "Integer"
        )

        print(
            f"Created {len(self.assignment_vars)} binary assignment variables."
        )
        print(
            f"Created {len(self.completion_vars)} bay completion lower bounds."
        )
        print(
            f"Created {len(self.vessel_completion_vars)} vessel completion lower bounds."
        )

    def add_objective_function(self):
        """Minimize the latest vessel completion time."""
        print("Adding the makespan objective...")
        self.model += self.makespan_var, "Minimize_Makespan"

    def add_constraints(self):
        """Add processing, capacity, spatial, and completion constraints."""
        print("Adding model constraints...")

        # A bay receives exactly its required number of processing periods.
        for v_idx, vessel in enumerate(self.vessels):
            for bay in vessel.bays:
                self.model += (
                    pulp.lpSum(
                        self.assignment_vars[(c, v_idx, bay.bay_id, t)]
                        for c in range(self.n_cranes)
                        for t in range(self.time_horizon)
                    )
                    == bay.processing_time,
                    f"bay_processing_{v_idx}_{bay.bay_id}",
                )

                # The work count is measured in periods, so concurrent cranes
                # must not count as multiple periods of work on the same bay.
                for t in range(self.time_horizon):
                    self.model += (
                        pulp.lpSum(
                            self.assignment_vars[(c, v_idx, bay.bay_id, t)]
                            for c in range(self.n_cranes)
                        )
                        <= 1,
                        f"bay_single_crane_{v_idx}_{bay.bay_id}_{t}",
                    )

        # A crane can process at most one bay in any period.
        for c in range(self.n_cranes):
            for t in range(self.time_horizon):
                self.model += (
                    pulp.lpSum(
                        self.assignment_vars[(c, v_idx, bay.bay_id, t)]
                        for v_idx, vessel in enumerate(self.vessels)
                        for bay in vessel.bays
                    )
                    <= 1,
                    f"crane_capacity_{c}_{t}",
                )

        # If two cranes are active together, their bay assignments must
        # preserve crane order and satisfy the minimum physical clearance.
        for t in range(self.time_horizon):
            for c1 in range(self.n_cranes):
                for c2 in range(c1 + 1, self.n_cranes):
                    for v1_idx, vessel1 in enumerate(self.vessels):
                        for bay1 in vessel1.bays:
                            for v2_idx, vessel2 in enumerate(self.vessels):
                                for bay2 in vessel2.bays:
                                    if (
                                        bay1.position + self.clearance_distance
                                        > bay2.position
                                    ):
                                        self.model += (
                                            self.assignment_vars[
                                                (c1, v1_idx, bay1.bay_id, t)
                                            ]
                                            + self.assignment_vars[
                                                (c2, v2_idx, bay2.bay_id, t)
                                            ]
                                            <= 1,
                                            "non_crossing_"
                                            f"{c1}_{c2}_{t}_"
                                            f"{v1_idx}_{bay1.bay_id}_"
                                            f"{v2_idx}_{bay2.bay_id}",
                                        )

        # Period t occupies [t, t + 1), so its completion time is t + 1.
        for v_idx, vessel in enumerate(self.vessels):
            for bay in vessel.bays:
                for t in range(self.time_horizon):
                    for c in range(self.n_cranes):
                        self.model += (
                            self.completion_vars[(v_idx, bay.bay_id)]
                            >= (t + 1)
                            * self.assignment_vars[
                                (c, v_idx, bay.bay_id, t)
                            ],
                            f"completion_def_{v_idx}_{bay.bay_id}_{c}_{t}",
                        )

        # Vessel completion bounds every bay completion; makespan bounds
        # every vessel completion.
        for v_idx, vessel in enumerate(self.vessels):
            for bay in vessel.bays:
                self.model += (
                    self.vessel_completion_vars[v_idx]
                    >= self.completion_vars[(v_idx, bay.bay_id)],
                    f"vessel_completion_{v_idx}_{bay.bay_id}",
                )

            self.model += (
                self.makespan_var >= self.vessel_completion_vars[v_idx],
                f"makespan_{v_idx}",
            )

        print(
            f"Added constraints for {self.n_vessels} vessels, "
            f"{self.n_cranes} cranes, and {self.n_bays} bays."
        )

    def solve_model(self, time_limit=60):
        """Solve the MILP and report whether optimality was certified."""
        print("Solving the mathematical model...")

        solver = pulp.PULP_CBC_CMD(msg=False, timeLimit=time_limit)
        self.model.solve(solver)
        status = pulp.LpStatus[self.model.status]

        if status == "Optimal":
            print("An optimal solution was found and certified.")
            print(
                "Optimal makespan: "
                f"{pulp.value(self.makespan_var)} time units"
            )
            return True

        if status == "Feasible":
            print(
                "A feasible solution was found, but optimality was not certified."
            )
            print(
                "Incumbent makespan: "
                f"{pulp.value(self.makespan_var)} time units"
            )
            return True

        print(f"No usable solution was reported. Solver status: {status}")
        return False

    def extract_solution(self):
        """Extract assignments and compute completion times from the schedule."""
        if self.model is None:
            return None

        if pulp.LpStatus[self.model.status] not in ["Optimal", "Feasible"]:
            return None

        solution = {
            "assignments": [],
            "schedule": {c: [] for c in range(self.n_cranes)},
            "completion_times": {},
            "vessel_completion": {},
            "makespan": pulp.value(self.makespan_var),
        }

        for c in range(self.n_cranes):
            for v_idx, vessel in enumerate(self.vessels):
                for bay in vessel.bays:
                    for t in range(self.time_horizon):
                        var_value = pulp.value(
                            self.assignment_vars[(c, v_idx, bay.bay_id, t)]
                        )
                        if var_value is not None and var_value > 0.5:
                            assignment = {
                                "crane": c,
                                "vessel": v_idx,
                                "bay": bay.bay_id,
                                "position": bay.position,
                                "time": t,
                                "processing_time": bay.processing_time,
                            }
                            solution["assignments"].append(assignment)
                            solution["schedule"][c].append(assignment)

        for c in range(self.n_cranes):
            solution["schedule"][c].sort(key=lambda item: item["time"])

        # Derive actual completion times from the final occupied period.
        for v_idx, vessel in enumerate(self.vessels):
            bay_completion_times = []
            for bay in vessel.bays:
                bay_times = [
                    item["time"]
                    for item in solution["assignments"]
                    if item["vessel"] == v_idx and item["bay"] == bay.bay_id
                ]
                if not bay_times:
                    return None
                completion_time = max(bay_times) + 1
                solution["completion_times"][(v_idx, bay.bay_id)] = (
                    completion_time
                )
                bay_completion_times.append(completion_time)

            solution["vessel_completion"][v_idx] = max(bay_completion_times)

        print(f"Extracted {len(solution['assignments'])} assigned periods.")
        print(f"Makespan: {solution['makespan']} time units")
        return solution

    def calculate_performance_metrics(self, solution):
        """Calculate workload, horizon utilization, and idle capacity."""
        if not solution:
            return {}

        total_processing_time = sum(
            bay.processing_time for vessel in self.vessels for bay in vessel.bays
        )
        metrics = {
            "makespan": solution["makespan"],
            "crane_utilization": {},
            "total_processing_time": total_processing_time,
            "total_idle_time": (
                self.n_cranes * self.time_horizon - total_processing_time
            ),
        }

        for c in range(self.n_cranes):
            busy_periods = len(solution["schedule"][c])
            metrics["crane_utilization"][c] = {
                "busy_periods": busy_periods,
                "total_periods": self.time_horizon,
                "utilization_rate": (
                    busy_periods / self.time_horizon
                    if self.time_horizon > 0
                    else 0
                ),
            }

        return metrics

    def visualize_solution(self, solution):
        """Plot crane schedules, vessel work, and utilization metrics."""
        if not solution:
            print("There is no solution to visualize.")
            return

        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
            2, 2, figsize=(16, 12)
        )
        fig.suptitle(
            "Integrated QCASP: Integer Programming Solution",
            fontsize=16,
            fontweight="bold",
        )

        colors = plt.cm.Set3(np.linspace(0, 1, max(self.n_vessels, 1)))
        vessel_legend = [
            Patch(facecolor=colors[v_idx], edgecolor="black", label=f"Vessel {v_idx + 1}")
            for v_idx in range(self.n_vessels)
        ]

        # Each rectangle spans one unit interval [t, t + 1).
        ax1.set_title("Crane Schedule (Gantt Chart)", fontweight="bold")
        ax1.set_xlabel("Time (period units)")
        ax1.set_ylabel("Crane")
        for c in range(self.n_cranes):
            for assignment in solution["schedule"][c]:
                vessel_idx = assignment["vessel"]
                start_time = assignment["time"]
                rect = Rectangle(
                    (start_time, c - 0.4),
                    1,
                    0.8,
                    facecolor=colors[vessel_idx],
                    edgecolor="black",
                    linewidth=1,
                )
                ax1.add_patch(rect)
                ax1.text(
                    start_time + 0.5,
                    c,
                    f"V{vessel_idx + 1}B{assignment['bay'] + 1}",
                    ha="center",
                    va="center",
                    fontsize=8,
                    fontweight="bold",
                )
        ax1.set_xlim(0, self.time_horizon)
        ax1.set_ylim(-0.5, self.n_cranes - 0.5)
        ax1.set_xticks(range(self.time_horizon + 1))
        ax1.set_yticks(range(self.n_cranes))
        ax1.set_yticklabels([f"Crane {c + 1}" for c in range(self.n_cranes)])
        ax1.grid(True, alpha=0.3)
        ax1.legend(handles=vessel_legend, title="Vessel")

        ax2.set_title("Vessel Operations by Bay", fontweight="bold")
        ax2.set_xlabel("Time (period units)")
        ax2.set_ylabel("Vessel")
        for v_idx, vessel in enumerate(self.vessels):
            bay_tasks = {}
            for assignment in solution["assignments"]:
                if assignment["vessel"] == v_idx:
                    bay_tasks.setdefault(assignment["bay"], []).append(
                        assignment["time"]
                    )

            for bay_offset, (bay_id, times) in enumerate(
                sorted(bay_tasks.items())
            ):
                y_position = v_idx + bay_offset * 0.15 - 0.3
                for t in times:
                    ax2.add_patch(
                        Rectangle(
                            (t, y_position),
                            1,
                            0.25,
                            facecolor=colors[v_idx],
                            edgecolor="black",
                            linewidth=1,
                        )
                    )
                first_time = min(times)
                ax2.text(
                    first_time + 0.5,
                    y_position + 0.125,
                    f"Bay {bay_id + 1}",
                    ha="center",
                    va="center",
                    fontsize=8,
                )

        ax2.set_xlim(0, self.time_horizon)
        ax2.set_ylim(-0.5, self.n_vessels - 0.5)
        ax2.set_xticks(range(self.time_horizon + 1))
        ax2.set_yticks(range(self.n_vessels))
        ax2.set_yticklabels(
            [f"Vessel {v_idx + 1}" for v_idx in range(self.n_vessels)]
        )
        ax2.grid(True, alpha=0.3)
        ax2.legend(handles=vessel_legend, title="Vessel")

        metrics = self.calculate_performance_metrics(solution)
        crane_ids = list(metrics["crane_utilization"])
        utilization_rates = [
            metrics["crane_utilization"][c]["utilization_rate"]
            for c in crane_ids
        ]
        ax3.set_title("Crane Utilization over the Horizon", fontweight="bold")
        ax3.set_xlabel("Crane")
        ax3.set_ylabel("Utilization rate")
        bars = ax3.bar(
            crane_ids,
            utilization_rates,
            color="skyblue",
            edgecolor="navy",
            linewidth=1,
        )
        ax3.set_ylim(0, 1)
        ax3.set_xticks(crane_ids)
        ax3.set_xticklabels([f"Crane {c + 1}" for c in crane_ids])
        for bar, rate in zip(bars, utilization_rates):
            ax3.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.02,
                f"{rate:.1%}",
                ha="center",
                va="bottom",
            )
        ax3.grid(True, alpha=0.3, axis="y")

        ax4.set_title("Performance Summary", fontweight="bold")
        ax4.axis("off")
        makespan = metrics["makespan"]
        efficiency = (
            metrics["total_processing_time"] / (makespan * self.n_cranes)
            if makespan and self.n_cranes
            else 0
        )
        metrics_text = (
            "PERFORMANCE INDICATORS\n"
            "\n"
            f"Makespan: {makespan} time units\n"
            "Total processing time: "
            f"{metrics['total_processing_time']} time units\n"
            "Idle crane capacity over the full horizon: "
            f"{metrics['total_idle_time']} crane-periods\n"
            f"Makespan-based system efficiency: {efficiency:.1%}\n"
            "\n"
            "Crane utilization over the full horizon:\n"
        )
        for c in range(self.n_cranes):
            util = metrics["crane_utilization"][c]
            metrics_text += (
                f"Crane {c + 1}: {util['utilization_rate']:.1%} "
                f"({util['busy_periods']}/{util['total_periods']} periods)\n"
            )
        metrics_text += (
            "\nModel: mixed-integer linear program\n"
            f"Binary assignment variables: {len(self.assignment_vars)}\n"
            "Integer completion-related variables: "
            f"{len(self.completion_vars) + len(self.vessel_completion_vars) + 1}\n"
            f"Constraints: {len(self.model.constraints)}\n"
            f"Solver status: {pulp.LpStatus[self.model.status]}\n"
            "Solver: CBC (COIN-OR)"
        )
        ax4.text(
            0.05,
            0.95,
            metrics_text,
            transform=ax4.transAxes,
            fontsize=10,
            verticalalignment="top",
            fontfamily="monospace",
            bbox={
                "boxstyle": "round,pad=0.5",
                "facecolor": "lightgray",
                "alpha": 0.8,
            },
        )

        plt.tight_layout()
        plt.show()

    def print_solution_summary(self, solution):
        """Print the schedule, completion times, and model checks."""
        if not solution:
            print("No solution is available.")
            return

        print("\n" + "=" * 72)
        print("INTEGRATED QCASP: INTEGER PROGRAMMING SOLUTION")
        print("=" * 72)
        print("\nProblem instance:")
        print(f"  Vessels: {self.n_vessels}")
        print(f"  Cranes: {self.n_cranes}")
        print(f"  Bays: {self.n_bays}")
        print(f"  Planning horizon: {self.time_horizon} periods")
        print(f"  Minimum clearance: {self.clearance_distance} position units")

        print("\nSolution:")
        print(f"  Makespan: {solution['makespan']} time units")
        print(f"  Assigned processing periods: {len(solution['assignments'])}")

        for c in range(self.n_cranes):
            schedule = solution["schedule"][c]
            print(f"\nCrane {c + 1}:")
            if not schedule:
                print("  No assignments (idle)")
            for assignment in schedule:
                print(
                    f"  Period {assignment['time']}: "
                    f"Vessel {assignment['vessel'] + 1}, "
                    f"Bay {assignment['bay'] + 1} "
                    f"(position {assignment['position']})"
                )

        print("\nCompletion times calculated from assigned periods:")
        for v_idx in range(self.n_vessels):
            print(
                f"  Vessel {v_idx + 1}: "
                f"{solution['vessel_completion'][v_idx]} time units"
            )

        metrics = self.calculate_performance_metrics(solution)
        average_utilization = np.mean(
            [
                item["utilization_rate"]
                for item in metrics["crane_utilization"].values()
            ]
        )
        efficiency = (
            metrics["total_processing_time"]
            / (metrics["makespan"] * self.n_cranes)
            if metrics["makespan"] and self.n_cranes
            else 0
        )
        print("\nPerformance metrics:")
        print(f"  Makespan-based system efficiency: {efficiency:.1%}")
        print(f"  Average crane utilization: {average_utilization:.1%}")
        print(
            "  Total processing time: "
            f"{metrics['total_processing_time']} time units"
        )
        print(
            "  Idle crane capacity over the full horizon: "
            f"{metrics['total_idle_time']} crane-periods"
        )

        # These checks validate the extracted schedule against the model's
        # processing, capacity, order, and clearance rules.
        for v_idx, vessel in enumerate(self.vessels):
            for bay in vessel.bays:
                assigned_periods = [
                    item
                    for item in solution["assignments"]
                    if item["vessel"] == v_idx and item["bay"] == bay.bay_id
                ]
                assert len(assigned_periods) == bay.processing_time
                assert len({item["time"] for item in assigned_periods}) == len(
                    assigned_periods
                )

        for c in range(self.n_cranes):
            crane_times = [
                item["time"] for item in solution["schedule"][c]
            ]
            assert len(crane_times) == len(set(crane_times))

        for t in range(self.time_horizon):
            active_assignments = [
                item
                for item in solution["assignments"]
                if item["time"] == t
            ]
            for first in active_assignments:
                for second in active_assignments:
                    if first["crane"] < second["crane"]:
                        assert (
                            first["position"] + self.clearance_distance
                            <= second["position"]
                        )

        actual_makespan = max(solution["vessel_completion"].values())
        assert actual_makespan <= solution["makespan"] + 1e-6
        print("\nSchedule checks passed for processing, capacity, order, and clearance.")
        print("=" * 72)

## Define the example instance

The data below specify two vessels with two bays each and three cranes. Bay positions determine the spatial order and clearance checks. Processing times are measured in whole time periods. The horizon is eight periods, which provides the model's available scheduling window; it is not a claim that the optimal schedule must use every period.

In [ ]:
print("Creating the example instance...")
print("Problem: 2 vessels, 3 cranes, 4 bays, and 8 time periods")

vessel1 = Vessel(
    vessel_id=0,
    arrival_time=0,
    due_date=10,
    bays=[
        Bay(vessel_id=0, bay_id=0, position=10, processing_time=2),
        Bay(vessel_id=0, bay_id=1, position=30, processing_time=3),
    ],
)

vessel2 = Vessel(
    vessel_id=1,
    arrival_time=0,
    due_date=10,
    bays=[
        Bay(vessel_id=1, bay_id=0, position=50, processing_time=1),
        Bay(vessel_id=1, bay_id=1, position=70, processing_time=2),
    ],
)

cranes = [
    Crane(crane_id=0, initial_position=0),
    Crane(crane_id=1, initial_position=20),
    Crane(crane_id=2, initial_position=40),
]

vessels = [vessel1, vessel2]
time_horizon = 8
clearance_distance = 2

print("\nInstance summary:")
print("  Vessel 1: positions [10, 30], processing times [2, 3]")
print("  Vessel 2: positions [50, 70], processing times [1, 2]")
print("  Crane initial positions: [0, 20, 40]")
print(f"  Planning horizon: {time_horizon} periods")
print(f"  Minimum clearance: {clearance_distance} position units")

## Build and solve the model

The calls follow the model's construction order: create variables, add the objective, add constraints, and then invoke CBC. A returned solution is extracted from the binary assignment variables. The printed completion times are computed from the last period assigned to each bay, using the interval convention that period \(t\) ends at \(t+1\).

In [ ]:
qcasp = IntegratedQCASP(
    vessels,
    cranes,
    time_horizon,
    clearance_distance,
)

qcasp.create_mathematical_model()
qcasp.add_objective_function()
qcasp.add_constraints()

if qcasp.solve_model(time_limit=30):
    solution = qcasp.extract_solution()

    if solution:
        qcasp.print_solution_summary(solution)
        qcasp.visualize_solution(solution)
    else:
        print("A solver status was reported, but no complete schedule was extracted.")
else:
    print("The model did not produce a usable solution within the solver run.")

## Interpreting the solution

The assignment variables describe which crane processes which bay in each period. Read the Gantt chart horizontally to follow a crane through time; each rectangle occupies one complete period. The vessel view groups those periods by vessel and bay. A bay may have interruptions because the formulation requires a total processing amount but does not require its work periods to be consecutive.

The makespan is the latest completion time across vessels. Crane utilization in the summary is measured over the full planning horizon, while makespan-based system efficiency compares total required processing time with the maximum crane capacity available up to the makespan. Idle crane capacity is reported over the entire planning horizon, so it is expressed in crane-periods. These measures describe different denominators and should not be treated as interchangeable.

The spatial constraints compare bay positions for pairs of cranes that are simultaneously active. They enforce the modeled order and clearance, but do not represent travel between successive assignments or the location of an idle crane. The model's conclusions should therefore be interpreted within the scope and assumptions stated above. An optimal solver status certifies the minimum makespan for this mathematical model and this instance; it does not imply that omitted operational details have been accounted for.